## [WIP] DuckDB with Iceberg Example

In [ ]:
# https://gist.github.com/eedduuar/ab8e2f4ea1934bd2a15f595b0c6da8c5

In [81]:
import os
from pathlib import Path

import duckdb
from pyiceberg.catalog.sql import SqlCatalog

In [82]:
catalog = SqlCatalog(
    "default",
    **{
        "uri": "sqlite:///.warehouse/pyiceberg_catalog.db",
        "warehouse": "file://.warehouse",
    },
)

In [83]:
# Namepsace is a group of tables.

catalog.create_namespace_if_not_exists(
    namespace="example",
    # properties={},
)

In [84]:
catalog.list_namespaces()

[('example',)]

In [85]:
db = duckdb.connect(
    database=Path(os.environ["DUCKDB_DIR"]) / "example.db",
    read_only=True,
)

In [86]:
import pyarrow

db_table = db.sql("SELECT * FROM source__mysql.users;").arrow()

# https://github.com/apache/iceberg-python/issues/541
timestamp_fields = [field.name for field in db_table.schema if pyarrow.types.is_timestamp(field.type)]
null_fields = [field.name for field in db_table.schema if pyarrow.types.is_null(field.type)]

fields = []
for field in db_table.schema:
    if field.name in timestamp_fields:
        fields.append(pyarrow.field(field.name, pyarrow.timestamp("us")))
    elif field.name in null_fields:
        fields.append(pyarrow.field(field.name, pyarrow.string()))
    else:
        fields.append(field)

db_table = db_table.cast(pyarrow.schema(fields))

catalog.create_table(identifier="example.first", schema=db_table.schema)

first(
  1: id: optional int,
  2: created_at: optional timestamp,
  3: updated_at: optional timestamp,
  4: role_id: optional int,
  5: name: optional string,
  6: email: optional string
),
partition by: [],
sort order: [],
snapshot: null

In [87]:
catalog.list_tables(namespace="example")

[('example', 'first')]

In [88]:
warehouse_table = catalog.load_table("example.first")

In [89]:
# https://juhache.substack.com/p/iceberg-single-node-engines

db_table = db.sql("SELECT * FROM source__mysql.users;").arrow()

fields = []
for field in db_table.schema:
    if field.name in timestamp_fields:
        fields.append(pyarrow.field(field.name, pyarrow.timestamp("us")))
    elif field.name in null_fields:
        fields.append(pyarrow.field(field.name, pyarrow.string()))
    else:
        fields.append(field)

db_table = db_table.cast(pyarrow.schema(fields))

warehouse_table.overwrite(db_table)

/home/okeyaki/projects/github.com/okeyaki/smalldata/.venv/lib/python3.13/site-packages/pyiceberg/table/__init__.py:558: UserWarning: Delete operation did not match any records
  warnings.warn("Delete operation did not match any records")


In [90]:
db.close()

In [101]:
warehouse_table.scan().to_pandas()

,id,created_at,updated_at,role_id,name,email
0,1,2024-12-11 04:41:00,2024-12-11 04:41:00,1,Alice,alice@localhost
1,2,2024-12-11 04:41:00,2024-12-11 04:41:00,1,Bob,bob@localhost
2,3,2024-12-11 04:41:00,2024-12-11 04:41:00,1,Charlie,charlie@localhost
3,4,2024-12-11 04:41:00,2024-12-11 04:41:00,2,David,david@localhost
4,5,2024-12-11 04:41:00,2024-12-11 04:41:00,2,Eve,eve@localhost
5,6,2024-12-11 04:41:00,2024-12-11 04:41:00,2,Frank,frank@localhost
6,7,2024-12-11 04:41:00,2024-12-11 04:41:00,2,Grace,grace@localhost
7,8,2024-12-11 04:41:00,2024-12-11 04:41:00,2,Heidi,heidi@localhost
8,9,2024-12-11 04:41:00,2024-12-11 04:41:00,2,Ivan,ivan@localhost
9,10,2024-12-11 04:41:00,2024-12-11 04:41:00,2,Judy,judy@localhost


In [ ]:
iceberg_db.close()